In [1]:
!pip install optuna


   -------------------- ------------------- 2/4 [alembic]
   -------------------- ------------------- 2/4 [alembic]
   -------------------- ------------------- 2/4 [alembic]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   ---------------------------------------- 4/4 [optuna]



In [8]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score



url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']


In [3]:
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [6]:
import numpy as np
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)
df.fillna(df.mean(), inplace=True)

print(df.isnull().sum())




Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [7]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')



Training set shape: (614, 8)
Test set shape: (154, 8)


In [14]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators',50,350)
    max_depth = trial.suggest_int('max_depth',3,20)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score




In [15]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler()) #int this maximize the accuracy using tpe sampler 
study.optimize(objective, n_trials=60)

[I 2026-10-17 23:26:02,632] A new study created in memory with name: no-name-3a6f0d4a-3e68-4064-b664-80191ecbe470
[I 2026-10-17 23:26:04,336] Trial 0 finished with value: 0.771975131516021 and parameters: {'n_estimators': 309, 'max_depth': 12}. Best is trial 0 with value: 0.771975131516021.
[I 2026-10-17 23:26:04,745] Trial 1 finished with value: 0.7507970667941973 and parameters: {'n_estimators': 84, 'max_depth': 3}. Best is trial 0 with value: 0.771975131516021.
[I 2026-10-17 23:26:06,289] Trial 2 finished with value: 0.7752271640363463 and parameters: {'n_estimators': 275, 'max_depth': 11}. Best is trial 2 with value: 0.7752271640363463.
[I 2026-10-17 23:26:07,986] Trial 3 finished with value: 0.7671050534034752 and parameters: {'n_estimators': 313, 'max_depth': 8}. Best is trial 2 with value: 0.7752271640363463.
[I 2026-10-17 23:26:09,511] Trial 4 finished with value: 0.7654551251394867 and parameters: {'n_estimators': 322, 'max_depth': 4}. Best is trial 2 with value: 0.77522716403

In [16]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7866491311971943
Best hyperparameters: {'n_estimators': 115, 'max_depth': 14}


In [17]:
from sklearn.metrics import accuracy_score
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.76
